In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'C:\Users\devpa\OneDrive\Desktop\Backup\Jarvis_V2\jarvis_intents_2000.csv')
print( df.sample(10))

                           command           intent
1141         check whatsapp for me    open_whatsapp
870            math calculator now  open_calculator
1735           open spotify please     open_spotify
824                 run calculator  open_calculator
321        please search on google    search_google
719               open text editor     open_notepad
1488  open github dashboard for me      open_github
1185             whatsapp open now    open_whatsapp
811   hey jarvis launch calculator  open_calculator
1679          please spotify music     open_spotify


In [3]:
df['intent'].value_counts()

intent
get_time           200
search_google      200
search_youtube     200
open_notepad       200
open_calculator    200
open_whatsapp      200
open_linkedin      200
open_github        200
open_spotify       200
exit               200
Name: count, dtype: int64

In [4]:
df.describe()

,command,intent
count,2000,2000
unique,650,10
top,search on google,get_time
freq,14,200


In [5]:
df['intent_number'] = df['intent'].map({
    'get_time': 0,
    'search_google': 1,
    'search_youtube': 2,
    'open_notepad': 3,
    'open_calculator': 4,
    'open_whatsapp': 5,
    'open_linkedin': 6,
    'open_github': 7,
    'open_spotify': 8,
    'exit': 9
})
print( df.sample(10))

                        command           intent  intent_number
1420   hey jarvis github please      open_github              7
80          current time please         get_time              0
670   hey jarvis notepad please     open_notepad              3
522              youtube search   search_youtube              2
493        can you play youtube   search_youtube              2
1794   hey jarvis spotify music     open_spotify              8
839             open calculator  open_calculator              4
240           look up on google    search_google              1
869         math calculator now  open_calculator              4
1522        start github for me      open_github              7


In [6]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

stopwords = list(STOP_WORDS)
print(stopwords)

['few', 'are', 'therein', 'would', 'but', '‘m', 'formerly', 'somehow', 'whom', 'themselves', 'hereby', 'side', 'unless', 'yourself', "'re", 'first', 'along', 'your', '‘ve', 'nowhere', 'indeed', "'s", 'everyone', 'everything', 'these', 'although', 'used', 'due', 'which', 'himself', 'before', 'elsewhere', 'move', 'same', 'in', 'give', 'together', 're', "'m", 'we', 'after', 'through', 'yours', 'and', 'does', 'five', 'out', 'less', 'really', 'itself', 'becomes', 'still', 'call', 'enough', 'my', 'take', 'once', 'this', 'or', 'into', 'n‘t', 'using', 'namely', 'can', 'a', 'wherein', 'between', 'thereafter', 'toward', 'though', 'just', 'seeming', 'thereupon', 'that', 'fifty', 'go', 'any', 'whether', 'was', 'with', 'here', 'done', 'where', '’re', '’ll', 'amount', 'how', 'well', 'own', 'two', 'regarding', 'least', 'across', 'neither', 'ten', 'whole', 'n’t', 'latter', 'seem', '‘s', 'made', 'nothing', 'rather', 'throughout', 'all', 'will', "'d", 'one', 'its', 'therefore', 'you', 'meanwhile', 'coul

In [7]:
df['command_without_stopwords'] = df['command'].apply(lambda x: ' '.join([word for word in x.split() if word not in stopwords]))

In [8]:
df.head(10)

,command,intent,intent_number,command_without_stopwords
0,jarvis time,get_time,0,jarvis time
1,time now please,get_time,0,time
2,what's the time,get_time,0,what's time
3,hey jarvis current time,get_time,0,hey jarvis current time
4,hey jarvis what's the time,get_time,0,hey jarvis what's time
5,what's the time,get_time,0,what's time
6,what time is it now,get_time,0,time
7,time please please,get_time,0,time
8,what's the time for me,get_time,0,what's time
9,hey jarvis what's the time,get_time,0,hey jarvis what's time


In [9]:
from spacy.lang.en.stop_words import STOP_WORDS
stopwords = list(STOP_WORDS)
df['command_without_stopwords'] = df['command'].apply(
    lambda x: ' '.join([word for word in x.split() if word.lower() not in stopwords])
)

# 4️⃣ Convert to embeddings
nlp = spacy.load('en_core_web_md')
df['command_vector'] = df['command_without_stopwords'].apply(lambda x: nlp(x).vector)

In [10]:
df.sample(10)


,command,intent,intent_number,command_without_stopwords,command_vector
1123,start my whatsapp,open_whatsapp,5,start whatsapp,"[-0.71024, 0.09702999, -0.0023449957, -0.10786..."
397,google search,search_google,1,google search,"[-0.679885, 0.57078004, -0.32488, 0.20802, -0...."
1207,linkedin profile,open_linkedin,6,linkedin profile,"[-0.856025, 0.79717, -0.27325, -0.0053199977, ..."
1265,linkedin please,open_linkedin,6,linkedin,"[-1.038, 1.0544, -0.16399, -0.17738, 0.25153, ..."
1975,can you terminate,exit,9,terminate,"[-0.76601, 0.46106, -0.25536, 0.074322, -0.366..."
58,hey jarvis tell me the time,get_time,0,hey jarvis tell time,"[-0.6738875, 0.037297495, -0.061524924, -0.301..."
1394,start linkedin for me,open_linkedin,6,start linkedin,"[-0.885755, 0.73415995, -0.303245, -0.23432499..."
1003,start whatsapp,open_whatsapp,5,start whatsapp,"[-0.71024, 0.09702999, -0.0023449957, -0.10786..."
940,open calc for me,open_calculator,4,open calc,"[-0.66603994, 0.19411999, -0.2113365, 0.316775..."
1024,hey jarvis open whatsapp,open_whatsapp,5,hey jarvis open whatsapp,"[-0.654875, -0.0947225, 0.16701674, -0.0559362..."


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.command_vector.values,
    df.intent_number,
    test_size=0.2,
    random_state=42
)

In [12]:
X_test.shape

(400,)

In [13]:
import numpy as np
X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)

In [14]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

# No scaling needed — spaCy vectors are already normalized enough.
clf = LinearSVC()

clf.fit(X_train_2d, y_train)

y_pred = clf.predict(X_test_2d)

print("Accuracy:", accuracy_score(y_test, y_pred)*100)
print("\nReport:\n", classification_report(y_test, y_pred))

joblib.dump(clf, "intent_model_v2.pkl")
print("Model saved as intent_model_v2.pkl")


Accuracy: 98.25

Report:
               precision    recall  f1-score   support

           0       0.84      1.00      0.91        36
           1       1.00      0.91      0.96        47
           2       1.00      1.00      1.00        46
           3       1.00      1.00      1.00        36
           4       1.00      1.00      1.00        34
           5       1.00      1.00      1.00        34
           6       1.00      1.00      1.00        33
           7       1.00      1.00      1.00        44
           8       1.00      1.00      1.00        48
           9       1.00      0.93      0.96        42

    accuracy                           0.98       400
   macro avg       0.98      0.98      0.98       400
weighted avg       0.99      0.98      0.98       400

Model saved as intent_model_v2.pkl


In [15]:
import joblib
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import numpy as np

# 1️⃣ Load saved model and scaler
stack_clf = joblib.load("jarvis_intent_model_stack.pkl")
scaler = joblib.load("jarvis_scaler_stack.pkl")

# 2️⃣ Load spaCy model
nlp = spacy.load('en_core_web_md')

# 3️⃣ Define stopwords
stopwords = list(STOP_WORDS)

# 4️⃣ Map intent numbers back to intent names
intent_map = {
    0: 'get_time',
    1: 'search_google',
    2: 'search_youtube',
    3: 'open_notepad',
    4: 'open_calculator',
    5: 'open_whatsapp',
    6: 'open_linkedin',
    7: 'open_github',
    8: 'open_spotify',
    9: 'exit'
}

# 5️⃣ Function to preprocess and predict intent
def predict_intent(command):
    # Remove stopwords
    command_clean = ' '.join([word for word in command.split() if word.lower() not in stopwords])
    
    # Convert to embedding
    command_vector = nlp(command_clean).vector.reshape(1, -1)
    
    # Scale vector
    command_scaled = scaler.transform(command_vector)
    
    # Predict intent number
    intent_number = stack_clf.predict(command_scaled)[0]
    
    # Map number to intent name
    return intent_map[intent_number]

# 6️⃣ Test commands
test_commands = [
    "Open Spotify",
    "What time is it?",
    "Launch Notepad",
    "Search Python tutorials on Google",
    "Open my GitHub profile",
    "Exit Jarvis",
    "Tell me the current time",
    "Close the assistant"
]

# 7️⃣ Run predictions
for cmd in test_commands:
    intent_name = predict_intent(cmd)
    print(f"Command: '{cmd}' -> Predicted Intent: {intent_name}")
    print (accuracy_score)


Command: 'Open Spotify' -> Predicted Intent: open_spotify
<function accuracy_score at 0x000002C400870A60>
Command: 'What time is it?' -> Predicted Intent: get_time
<function accuracy_score at 0x000002C400870A60>
Command: 'Launch Notepad' -> Predicted Intent: open_notepad
<function accuracy_score at 0x000002C400870A60>
Command: 'Search Python tutorials on Google' -> Predicted Intent: search_google
<function accuracy_score at 0x000002C400870A60>
Command: 'Open my GitHub profile' -> Predicted Intent: open_github
<function accuracy_score at 0x000002C400870A60>
Command: 'Exit Jarvis' -> Predicted Intent: exit
<function accuracy_score at 0x000002C400870A60>
Command: 'Tell me the current time' -> Predicted Intent: get_time
<function accuracy_score at 0x000002C400870A60>
Command: 'Close the assistant' -> Predicted Intent: exit
<function accuracy_score at 0x000002C400870A60>
